# Model Comparison & Evaluation\n\nCompare all 3 models on common metrics to select the best one for the web app.\n\n| Model | Type | Classification | Segmentation | Speed |\n|-------|------|---------------|-------------|-------|\n| Watershed | Classical CV | No (binary only) | Binary masks | Very fast |\n| YOLOv8s-seg | DL (single-stage) | Yes (9 classes) | Instance masks | Fast |\n| Mask R-CNN | DL (two-stage) | Yes (9 classes) | Instance masks | Moderate |

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================
# PATHS
# =========================

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
RUNS_DIR = os.path.join(BASE_DIR, "runs")
OUTPUT_DIR = os.path.join(RUNS_DIR, "evaluation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load results from each model
watershed_path = os.path.join(RUNS_DIR, "watershed", "watershed_results.json")
yolo_path = os.path.join(RUNS_DIR, "yolov8_seg", "taco_v2", "yolov8_seg_results.json")
maskrcnn_path = os.path.join(RUNS_DIR, "mask_rcnn", "mask_rcnn_results.json")

results = {}

for name, path in [("Watershed", watershed_path), ("YOLOv8s-seg", yolo_path), ("Mask R-CNN", maskrcnn_path)]:
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
        print(f"Loaded: {name}")
    else:
        print(f"NOT FOUND: {name} ({path})")

print(f"\nModels loaded: {list(results.keys())}")

## Summary Metrics Table

In [ ]:
# =========================
# BUILD COMPARISON TABLE
# =========================

rows = []

for model_name, data in results.items():
    m = data["metrics"]
    row = {
        "Model": model_name,
        "Box mAP@50": m.get("box_map50", "N/A"),
        "Box mAP@50-95": m.get("box_map50_95", "N/A"),
        "Mask mAP@50": m.get("mask_map50", "N/A"),
        "Mask mAP@50-95": m.get("mask_map50_95", "N/A"),
        "Binary IoU": m.get("binary_iou", "N/A"),
        "Counting MAE": m.get("counting_mae", "N/A"),
        "Count +/-1 (%)": m.get("counting_within_1", "N/A"),
        "Count +/-3 (%)": m.get("counting_within_3", "N/A"),
        "Avg Speed (ms)": m.get("avg_inference_ms", "N/A"),
    }
    rows.append(row)

df = pd.DataFrame(rows).set_index("Model")

# Format numeric columns
for col in df.columns:
    df[col] = df[col].apply(lambda x: f"{x:.4f}" if isinstance(x, float) and x < 10 else 
                            (f"{x:.1f}" if isinstance(x, float) else x))

print("=" * 80)
print("MODEL COMPARISON - TEST SET RESULTS")
print("=" * 80)
print(df.to_string())
print("=" * 80)

## Comparison Charts

In [ ]:
# =========================
# COMPARISON BAR CHARTS
# =========================

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

model_names = list(results.keys())
colors_map = {"Watershed": "#2ecc71", "YOLOv8s-seg": "#3498db", "Mask R-CNN": "#e74c3c"}
bar_colors = [colors_map.get(n, "#95a5a6") for n in model_names]

# 1. mAP@50 (DL models only)
ax = axes[0, 0]
dl_models = [n for n in model_names if n != "Watershed"]
dl_colors = [colors_map[n] for n in dl_models]
map50_vals = [float(results[n]["metrics"].get("box_map50", 0)) for n in dl_models]
bars = ax.bar(dl_models, map50_vals, color=dl_colors)
ax.set_title("Box mAP@50", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
for bar, val in zip(bars, map50_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f"{val:.3f}", ha="center", fontsize=12)

# 2. Mask mAP@50 (DL models only)
ax = axes[0, 1]
mask_vals = [float(results[n]["metrics"].get("mask_map50", 0)) for n in dl_models]
bars = ax.bar(dl_models, mask_vals, color=dl_colors)
ax.set_title("Mask mAP@50", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
for bar, val in zip(bars, mask_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f"{val:.3f}", ha="center", fontsize=12)

# 3. Counting MAE (all models, lower is better)
ax = axes[0, 2]
mae_vals = [float(results[n]["metrics"]["counting_mae"]) for n in model_names]
bars = ax.bar(model_names, mae_vals, color=bar_colors)
ax.set_title("Counting MAE (lower is better)", fontsize=14, fontweight="bold")
for bar, val in zip(bars, mae_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f"{val:.2f}", ha="center", fontsize=12)

# 4. Counting accuracy within +/-1 (all models)
ax = axes[1, 0]
acc_vals = [float(results[n]["metrics"]["counting_within_1"]) for n in model_names]
bars = ax.bar(model_names, acc_vals, color=bar_colors)
ax.set_title("Counting Accuracy (+/-1)", fontsize=14, fontweight="bold")
ax.set_ylim(0, 100)
ax.set_ylabel("%")
for bar, val in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f"{val:.1f}%", ha="center", fontsize=12)

# 5. Inference speed (all models, lower is better)
ax = axes[1, 1]
speed_vals = [float(results[n]["metrics"]["avg_inference_ms"]) for n in model_names]
bars = ax.bar(model_names, speed_vals, color=bar_colors)
ax.set_title("Avg Inference Time (lower is better)", fontsize=14, fontweight="bold")
ax.set_ylabel("ms")
for bar, val in zip(bars, speed_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f"{val:.0f}ms", ha="center", fontsize=12)

# 6. Counting accuracy within +/-3 (all models)
ax = axes[1, 2]
acc3_vals = [float(results[n]["metrics"]["counting_within_3"]) for n in model_names]
bars = ax.bar(model_names, acc3_vals, color=bar_colors)
ax.set_title("Counting Accuracy (+/-3)", fontsize=14, fontweight="bold")
ax.set_ylim(0, 100)
ax.set_ylabel("%")
for bar, val in zip(bars, acc3_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f"{val:.1f}%", ha="center", fontsize=12)

plt.suptitle("Model Comparison - All Metrics", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_charts.png"), dpi=150, bbox_inches="tight")
plt.show()

## Per-Image Counting Error Distribution

In [ ]:
# =========================
# COUNTING ERROR DISTRIBUTION
# =========================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (model_name, data) in enumerate(results.items()):
    errors = [r["count_error"] for r in data["per_image"]]
    
    ax = axes[idx]
    ax.hist(errors, bins=range(0, max(errors) + 2), alpha=0.7, 
            color=bar_colors[idx], edgecolor="black")
    ax.set_title(f"{model_name}\nMAE={np.mean(errors):.2f}", fontsize=13)
    ax.set_xlabel("Absolute Count Error")
    ax.set_ylabel("Number of Images")
    ax.axvline(np.mean(errors), color="red", linestyle="--", label=f"Mean={np.mean(errors):.2f}")
    ax.legend()

plt.suptitle("Counting Error Distribution per Model", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "counting_error_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## Predicted vs Ground Truth Count (Scatter Plot)

In [ ]:
# =========================
# PREDICTED VS GT COUNT SCATTER
# =========================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (model_name, data) in enumerate(results.items()):
    gt = [r["gt_count"] for r in data["per_image"]]
    pred = [r["pred_count"] for r in data["per_image"]]
    
    ax = axes[idx]
    ax.scatter(gt, pred, alpha=0.5, s=30, color=bar_colors[idx])
    
    max_val = max(max(gt), max(pred)) + 2
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=1, label="Perfect")
    ax.set_xlim(-0.5, max_val)
    ax.set_ylim(-0.5, max_val)
    ax.set_xlabel("Ground Truth Count")
    ax.set_ylabel("Predicted Count")
    ax.set_title(f"{model_name}", fontsize=13)
    ax.legend()
    ax.set_aspect("equal")

plt.suptitle("Predicted vs Ground Truth Object Count", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pred_vs_gt_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

## Model Selection Score for Web App\n\nWeighted scoring:\n- 40% mAP@50 (detection quality)\n- 20% counting accuracy (+/-1)\n- 20% mask quality (mask mAP or binary IoU)\n- 10% inference speed\n- 10% model size\n\nWatershed gets 0 for mAP and mask mAP since it can't classify. Its counting and speed scores are compared fairly.

In [ ]:
# =========================
# MODEL SELECTION SCORING
# =========================

# Approximate model sizes
model_sizes = {
    "Watershed": 0.001,     # Essentially 0 (no model file, just code)
    "YOLOv8s-seg": 23.0,   # ~23 MB
    "Mask R-CNN": 170.0,    # ~170 MB
}

def normalize(values, lower_is_better=False):
    """Normalize values to [0, 1]. Higher is better by default."""
    vals = np.array(values, dtype=float)
    if vals.max() == vals.min():
        return np.ones_like(vals) * 0.5
    if lower_is_better:
        return 1.0 - (vals - vals.min()) / (vals.max() - vals.min())
    return (vals - vals.min()) / (vals.max() - vals.min())

model_names_ordered = list(results.keys())

# Extract metrics
map50_scores = [float(results[n]["metrics"].get("box_map50", 0)) for n in model_names_ordered]
count_acc = [float(results[n]["metrics"]["counting_within_1"]) / 100 for n in model_names_ordered]
mask_quality = [float(results[n]["metrics"].get("mask_map50", results[n]["metrics"].get("binary_iou", 0))) 
                for n in model_names_ordered]
speeds = [float(results[n]["metrics"]["avg_inference_ms"]) for n in model_names_ordered]
sizes = [model_sizes[n] for n in model_names_ordered]

# Normalize
norm_map50 = normalize(map50_scores)
norm_count = normalize(count_acc)
norm_mask = normalize(mask_quality)
norm_speed = normalize(speeds, lower_is_better=True)
norm_size = normalize(sizes, lower_is_better=True)

# Weighted score
weights = {"mAP50": 0.4, "counting": 0.2, "mask": 0.2, "speed": 0.1, "size": 0.1}

final_scores = {}
print("=" * 70)
print("MODEL SELECTION SCORING")
print("=" * 70)
print(f"{'Model':<15} {'mAP50':>8} {'Count':>8} {'Mask':>8} {'Speed':>8} {'Size':>8} {'TOTAL':>8}")
print("-" * 70)

for i, name in enumerate(model_names_ordered):
    score = (weights["mAP50"] * norm_map50[i] +
             weights["counting"] * norm_count[i] +
             weights["mask"] * norm_mask[i] +
             weights["speed"] * norm_speed[i] +
             weights["size"] * norm_size[i])
    final_scores[name] = score
    
    print(f"{name:<15} {norm_map50[i]:>8.3f} {norm_count[i]:>8.3f} "
          f"{norm_mask[i]:>8.3f} {norm_speed[i]:>8.3f} {norm_size[i]:>8.3f} {score:>8.3f}")

print("=" * 70)

best_model = max(final_scores, key=final_scores.get)
print(f"\nRECOMMENDED MODEL FOR WEB APP: {best_model}")
print(f"Score: {final_scores[best_model]:.3f}")

# Radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
categories_radar = list(weights.keys())
N = len(categories_radar)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for i, name in enumerate(model_names_ordered):
    values = [norm_map50[i], norm_count[i], norm_mask[i], norm_speed[i], norm_size[i]]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=name, color=bar_colors[i])
    ax.fill(angles, values, alpha=0.1, color=bar_colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(["mAP@50", "Counting\nAccuracy", "Mask\nQuality", "Speed", "Model\nSize"], fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.set_title("Model Comparison Radar", fontsize=14, fontweight="bold", pad=20)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "radar_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## Final Summary

In [ ]:
# =========================
# FINAL SUMMARY
# =========================

print("=" * 70)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 70)

print("""
Model Rankings (by weighted score):
""")

for rank, (name, score) in enumerate(sorted(final_scores.items(), key=lambda x: x[1], reverse=True), 1):
    m = results[name]["metrics"]
    print(f"  #{rank} {name}")
    print(f"     Score: {score:.3f}")
    if "box_map50" in m:
        print(f"     Box mAP@50: {m['box_map50']:.4f}")
    if "mask_map50" in m:
        print(f"     Mask mAP@50: {m['mask_map50']:.4f}")
    if "binary_iou" in m:
        print(f"     Binary IoU: {m['binary_iou']:.4f}")
    print(f"     Counting MAE: {m['counting_mae']:.2f}")
    print(f"     Counting +/-1: {m['counting_within_1']:.1f}%")
    print(f"     Avg Speed: {m['avg_inference_ms']:.1f} ms")
    print()

print(f"RECOMMENDED FOR WEB APP: {best_model}")
print("=" * 70)

# Save final report
report = {
    "ranking": [
        {"rank": rank, "model": name, "score": score}
        for rank, (name, score) in enumerate(
            sorted(final_scores.items(), key=lambda x: x[1], reverse=True), 1
        )
    ],
    "recommended": best_model,
    "all_metrics": {name: data["metrics"] for name, data in results.items()},
}

with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
    json.dump(report, f, indent=2)

print(f"\nFinal report saved to: {OUTPUT_DIR}/final_report.json")
print(f"Comparison charts saved to: {OUTPUT_DIR}/")